# trapX LLM Advisory — **`advisory_any_bar.py`** (EXTERNAL users)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_any_bar_colab_external.ipynb)

**Run from any Google account** — no Drive mount required. This is the external-user edition of the latest `advisory_any_bar.py` tool. Given a **DATE** and **BAR_TIME** it:

1. Downloads that day's folder **read-only** from the owner's shared Google Drive (`gdown`, public — no mount, no login).
2. Gates on the jsonl, rebuilds the advisory input, and fires **one converged LLM call** — via your **own Gemini API key** (see Step 1 below), against the trapx-default `gemini` backend. The old owner-Apps-Script proxy still works if `TRAPX_LLM_PROXY` is set in the process env.
3. Sends the run's verbose log back to the **owner** for central collection (`external_user_logs/`).

**Bring your own key.** Create a **free** Gemini API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey) and add it as a Colab Secret named `GEMINI_API_KEY` (🔑 icon in the left sidebar → `+ Add new secret` → toggle *Notebook access* ON). Step 1 lifts it into the process env; the embedded trapx `LLMClient._call_gemini` reads it via the CHA-350/351 env-fallback chain (`GEMINI_API_KEY_ADVISORY` → `GEMINI_API_KEY`).

**Local-parity replay.** When the owner has dropped a `pg_snapshot_<date>.db` into the day folder (produced on the trapX box with `_export_pg_day_snapshot.py`), gdown pulls it down automatically and the embedded `advisory_any_bar.py` **auto-detects** it — the resulting log is byte-identical to a local real-Postgres replay (closes the CSV-only gap: run-cumulative floor/ceiling TRAP, option price-symmetry, jerk-family windowed OI). If the snapshot isn't in the day folder yet, the run still succeeds CSV-only (older behaviour) and the log carries an explicit divergence note.

**Not on Colab.** Breeze 1-second futures data. If the requested bar fires `topbottom_formation`, that one touchpoint's 1-sec sustain reads as unavailable in the log — the verdict still runs.

## ⬇ STEP 1 — Enter your NAME or EMAIL (required to run)
Identifies your run in the owner's central log folder. Until it's filled, the run + log steps **skip** (nothing happens).

In [ ]:
#@title ⬇ Your name or email — then Run all { run: "auto" }
EXTERNAL_USER = ""  #@param {type:"string"}
WHO = EXTERNAL_USER.strip()
if WHO:
    print('Hi', WHO + ' — your log will be saved as', WHO + '_<date>_<time>.log')
else:
    print('=' * 66)
    print('  ENTER YOUR NAME OR EMAIL in the box above, then Run all again.')
    print('  Until then, the run + log steps below SKIP — nothing happens.')
    print('=' * 66)

## 1. Install deps + lift your Gemini key from Colab Secrets
Pins `gdown==6.1.0` (Colab's bundled gdown can't list the >50-item shared folder). Reads `GEMINI_API_KEY` from Colab **Secrets** into the process env — the embedded `LLMClient._call_gemini` picks it up via the CHA-350/351 env chain (`GEMINI_API_KEY_ADVISORY` → `GEMINI_API_KEY`). `NVIDIA_API_KEY` and `ANTHROPIC_API_KEY` are also lifted if present, so you can flip `BACKEND` in Step 3 without re-uploading the key.

In [ ]:
# gdown==6.1.0: Colab ships an OLD gdown (5.x) whose folder parser can't
# list a >50-item folder; 6.1.0 can (needed to find the day folder).
!pip install -q 'gdown==6.1.0' requests openai anthropic google-genai python-dotenv langgraph langgraph-checkpoint-sqlite

import os
# Copy every recognised backend key from Colab Secrets → process env,
# so the embedded LLMClient (trapx CHA-361) finds them via the same
# fallback chain the live engine uses.
_KEYS = ('GEMINI_API_KEY', 'GEMINI_API_KEY_ADVISORY',
         'NVIDIA_API_KEY', 'ANTHROPIC_API_KEY', 'OPENROUTER_API_KEY')
try:
    from google.colab import userdata
    for _k in _KEYS:
        try:
            _v = userdata.get(_k)
        except Exception:
            _v = None
        if _v:
            os.environ[_k] = _v.strip()
except Exception:
    pass

def _mask(v):
    return f'set (…{v[-4:]})' if v else 'MISSING'

print('GEMINI_API_KEY   :', _mask(os.environ.get('GEMINI_API_KEY', '')))
print('NVIDIA_API_KEY   :', _mask(os.environ.get('NVIDIA_API_KEY', '')))
print('ANTHROPIC_API_KEY:', _mask(os.environ.get('ANTHROPIC_API_KEY', '')))

## 2. Write the embedded `advisory_any_bar.py` + `skills/` + `project/` + wrapper to disk
Decodes the base64 payloads into `/content/` — the latest tool, all skills, the minimal engine `project/` package (so the v5 layer + jerk/signal backbones recompute), and the `run_advisory.py` wrapper that softens `pg_connect()`'s exit + routes the LLM through the owner collector.

In [ ]:
import base64, json, pathlib

SCRIPT_B64  = "..."
SKILLS_B64  = "..."
PROJECT_B64 = "..."
WRAPPER_B64 = "IiIiQ29sYWIgd3JhcHBlciBhcm91bmQgYWR2aXNvcnlfYW55X2Jhci5weS4KClR3byByZXNwb25zaWJpbGl0aWVzOgoKMSkgU09GVEVOIHBnX2Nvbm5lY3QoKSBzbyB0aGUgQ1NWLW9ubHkgZmFsbGJhY2sgZW5nYWdlcyB3aGVuIENvbGFiIGhhcwogICBuZWl0aGVyIHJlYWwgUG9zdGdyZXMgbm9yIGEgcGdfc25hcHNob3QgaW4gdGhlIGRheSBmb2xkZXIuCjIpIE9QVElPTkFMTFkgcm91dGUgTExNQ2xpZW50LmNhbGwoKSB0aHJvdWdoIGFuIG93bmVyJ3MgQXBwcyBTY3JpcHQgcHJveHkKICAgd2hlbiBUUkFQWF9MTE1fUFJPWFkgaXMgc2V0LiBXaGVuIGl0J3MgTk9UIHNldCAodGhlIG5ldyBkZWZhdWx0IG5vdwogICB0aGF0IGdlbWluaSBpcyB0aGUgbGl2ZSBiYWNrZW5kKSwgdGhlIHJlYWwgTExNQ2xpZW50IGhhbmRsZXMgdGhlIExMTQogICBjYWxsIGRpcmVjdGx5IOKAlCByZWFkaW5nIHRoZSB1c2VyJ3Mgb3duIEdFTUlOSV9BUElfS0VZIGZyb20gdGhlCiAgIGVudmlyb25tZW50IHZpYSB0aGUgQ0hBLTM1MC8zNTEgZ2VtaW5pX2tleV9wb29sIGVudiBmYWxsYmFjay4KCk9sZCBmbG93IChwcmUtZ2VtaW5pLWRlZmF1bHQpOiB0aGUgb3duZXIgcmFuIGFuIEFwcHMgU2NyaXB0IHRoYXQgaGVsZCB0aGUKTlZJRElBIGtleSBzZXJ2ZXItc2lkZSBhbmQgZXZlcnkgZXh0ZXJuYWwgQ29sYWIgUE9TVGVkIHRocm91Z2ggaXQuCgpOZXcgZmxvdyAodGhpcyBmaWxlKTogdGhlIHVzZXIgcGFzdGVzIHRoZWlyIG93biBHRU1JTklfQVBJX0tFWSBpbnRvCkNvbGFiIFNlY3JldHMg4oCUIFN0ZXAgMSBsaWZ0cyBpdCBpbnRvIHRoZSBwcm9jZXNzIGVudiDigJQgYW5kIHRoZSByZWFsCkxMTUNsaWVudC5fY2FsbF9nZW1pbmkgdXNlcyBpdCBkaXJlY3RseS4gVGhlIHByb3h5IHBhdGggaXMgcHJlc2VydmVkCnVuZGVyIFRSQVBYX0xMTV9QUk9YWSBmb3IgYW55b25lIHN0aWxsIG9uIHRoZSBvbGQgb3duZXIta2V5IHNldHVwLgoKQ0hBLTM2MSBwaGFzZSA0QiBjb2xsYXBzZWQgZXZlcnkgcGVyLWJhY2tlbmQgdHJhbnNwb3J0IGludG8gYSBzaW5nbGUKYExMTUNsaWVudC5jYWxsKClgOyB0aGF0IGlzIHRoZSBvbmUgZW50cnkgcG9pbnQgd2UgaW50ZXJjZXB0LgoiIiIKaW1wb3J0IHN5cywgb3MKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUKaW1wb3J0IGFkdmlzb3J5X2FueV9iYXIgYXMgYWFiCmZyb20gcHJvamVjdC5sbG1fYWR2aXNvcnkuY2xpZW50IGltcG9ydCBMTE1DbGllbnQKCiMgMSkgQ29sYWIgaGFzIG5vIFBvc3RncmVzLiBwZ19jb25uZWN0KCkgcmFpc2VzIFN5c3RlbUV4aXQgb24gYSBmYWlsZWQKIyAgICBjb25uZWN0IChvciBhIG1pc3NpbmcgLS1wZy1zbmFwc2hvdCBmaWxlKSwgd2hpY2ggdGhlIHNjcmlwdCdzIHJlcGxheQojICAgIENTVi1vbmx5IGZhbGxiYWNrIChleGNlcHQgRXhjZXB0aW9uKSBjYW5ub3QgY2F0Y2guIENvbnZlcnQgaXQgdG8gYQojICAgIGNhdGNoYWJsZSBlcnJvciBzbyB0aGUgaW50ZW5kZWQgY29ubj1Ob25lIENTVi1vbmx5IHBhdGggcnVucyB3aGVuCiMgICAgbmVpdGhlciByZWFsIFBHIG5vciBhIHNuYXBzaG90IGlzIGF2YWlsYWJsZS4gV2hlbiBhZHZpc29yeSBhdXRvLWRldGVjdHMKIyAgICBhIHBnX3NuYXBzaG90X1lZWVlNTURELmRiIGluIHRoZSBkYXkgZm9sZGVyLCBwZ19jb25uZWN0IHJldHVybnMgdGhlCiMgICAgc3FsaXRlIHNoaW0gc3VjY2Vzc2Z1bGx5IOKAlCB0aGlzIHdyYXBwZXIgaXMgYSBuby1vcCBpbiB0aGF0IGNhc2UuCl9vcmlnX3BnID0gYWFiLnBnX2Nvbm5lY3QKZGVmIF9wZ19vcHRpb25hbCgpOgogICAgdHJ5OgogICAgICAgIHJldHVybiBfb3JpZ19wZygpCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigicG9zdGdyZXMgdW5hdmFpbGFibGUgb24gQ29sYWI7IENTVi1vbmx5OiAlcyIgJSBlKQphYWIucGdfY29ubmVjdCA9IF9wZ19vcHRpb25hbAoKCiMgMikgV2hlbiBUUkFQWF9MTE1fUFJPWFkgaXMgc2V0LCByb3V0ZSBMTE1DbGllbnQuY2FsbCgpIHRocm91Z2ggdGhlIG93bmVyJ3MKIyAgICBBcHBzIFNjcmlwdCBwcm94eSAoaG9sZHMgZXZlcnkgYmFja2VuZCBrZXkgc2VydmVyLXNpZGUsIGRpc3BhdGNoZXMgYnkKIyAgICBtb2RlbCBpZCkuIERldGVybWluaXN0aWMgcGFyYW1zICh0ZW1wZXJhdHVyZSAwLCBzZWVkIDQyKSBtYXRjaCB0aGUKIyAgICBkaXJlY3QgU0RLIHBhdGgsIHNvIHZlcmRpY3RzIGFyZSBpZGVudGljYWwuIFByb3h5IGFkYXB0ZXIgbWF0Y2hlcwojICAgIExMTUNsaWVudCdzIG5hdGl2ZSByZXNwb25zZSBzaGFwZSAoc2VlIGNsaWVudC5weTo2NTQpOgojICAgICAgeyJ0ZXh0Ijogc3RyLCAidXNhZ2UiOiBkaWN0LCAibGF0ZW5jeV9tcyI6IGZsb2F0LCAicmF3IjogZGljdH0KIwojICAgIFdoZW4gVFJBUFhfTExNX1BST1hZIGlzIFVOU0VUICh0aGUgZGVmYXVsdCBub3cgdGhhdCB1c2VycyBwcm92aWRlIHRoZWlyCiMgICAgb3duIEdFTUlOSV9BUElfS0VZKSwgdGhpcyBtb25rZXktcGF0Y2ggaXMgaW5lcnQg4oCUIHRoZSByZWFsIExMTUNsaWVudAojICAgIGRpc3BhdGNoZXMgdG8gd2hpY2hldmVyIGJhY2tlbmQgdGhlIG9wZXJhdG9yIHBpY2tlZCBhbmQgcmVhZHMgdGhlIGtleQojICAgIGZyb20gdGhlIGVudiAoZ2VtaW5pIOKGkiBHRU1JTklfQVBJX0tFWV9BRFZJU09SWSAvIEdFTUlOSV9BUElfS0VZIGNoYWluCiMgICAgdmlhIHRoZSBDSEEtMzUxIGdlbWluaV9rZXlfcG9vbCBlbnYgZmFsbGJhY2spLgpfTExNQ2xpZW50X2NhbGxfb3JpZyA9IExMTUNsaWVudC5jYWxsCgpkZWYgX3Byb3hpZWRfY2FsbChzZWxmLCBzeXN0ZW0sIHVzZXIsIGZvcm1hdD1Ob25lLCBtYXhfdG9rZW5zPU5vbmUpOgogICAgcHJveHkgPSBvcy5lbnZpcm9uLmdldCgiVFJBUFhfTExNX1BST1hZIiwgIiIpLnN0cmlwKCkKICAgIGlmIG5vdCBwcm94eToKICAgICAgICByZXR1cm4gX0xMTUNsaWVudF9jYWxsX29yaWcoc2VsZiwgc3lzdGVtLCB1c2VyLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3JtYXQ9Zm9ybWF0LCBtYXhfdG9rZW5zPW1heF90b2tlbnMpCgogICAgaW1wb3J0IHRpbWUKICAgIGltcG9ydCByZXF1ZXN0cwogICAgcGF5bG9hZCA9IHsiYWN0aW9uIjogImxsbSIsICJiYWNrZW5kIjogc2VsZi5iYWNrZW5kLCAibW9kZWwiOiBzZWxmLm1vZGVsLAogICAgICAgICAgICAgICAic3lzdGVtIjogc3lzdGVtLCAidXNlciI6IHVzZXIsICJ0ZW1wZXJhdHVyZSI6IDAuMCwgInNlZWQiOiA0Mn0KICAgIGlmIG1heF90b2tlbnMgaXMgbm90IE5vbmU6CiAgICAgICAgcGF5bG9hZFsibWF4X3Rva2VucyJdID0gbWF4X3Rva2VucwogICAgaWYgZm9ybWF0IGlzIG5vdCBOb25lOgogICAgICAgIHBheWxvYWRbImZvcm1hdCJdID0gZm9ybWF0CgogICAgYXR0ZW1wdHMgPSAzCiAgICBsYXN0X2V4YyA9IE5vbmUKICAgIGZvciBpIGluIHJhbmdlKGF0dGVtcHRzICsgMSk6CiAgICAgICAgdDAgPSBkYXRldGltZS5ub3coKQogICAgICAgIHRyeToKICAgICAgICAgICAgciA9IHJlcXVlc3RzLnBvc3QocHJveHksIGpzb249cGF5bG9hZCwgdGltZW91dD1zZWxmLnRpbWVvdXRfc2VjKQogICAgICAgICAgICBsYXRlbmN5X21zID0gKGRhdGV0aW1lLm5vdygpIC0gdDApLnRvdGFsX3NlY29uZHMoKSAqIDEwMDAuMAogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBkYXRhID0gci5qc29uKCkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTExNIHByb3h5IHJldHVybmVkIG5vbi1KU09OIChIVFRQICVzKTogJXMiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJSAoci5zdGF0dXNfY29kZSwgci50ZXh0WzoyMDBdKSkKICAgICAgICAgICAgaWYgbm90IGRhdGEuZ2V0KCJvayIsIEZhbHNlKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTExNIHByb3h5IGVycm9yOiAlcyIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAlIGRhdGEuZ2V0KCJlcnJvciIsICJ1bmtub3duIikpCiAgICAgICAgICAgIHVzYWdlID0gZGF0YS5nZXQoInVzYWdlIiwge30pIG9yIHt9CiAgICAgICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICAgICAidGV4dCI6IGRhdGEuZ2V0KCJ0ZXh0IiwgIiIpLAogICAgICAgICAgICAgICAgInVzYWdlIjogewogICAgICAgICAgICAgICAgICAgICJpbnB1dF90b2tlbnMiOiAgaW50KHVzYWdlLmdldCgiaW5wdXRfdG9rZW5zIiwgIHVzYWdlLmdldCgicHJvbXB0X3Rva2VucyIsIDApKSBvciAwKSwKICAgICAgICAgICAgICAgICAgICAib3V0cHV0X3Rva2VucyI6IGludCh1c2FnZS5nZXQoIm91dHB1dF90b2tlbnMiLCB1c2FnZS5nZXQoImNvbXBsZXRpb25fdG9rZW5zIiwgMCkpIG9yIDApLAogICAgICAgICAgICAgICAgICAgICJjYWNoZV9yZWFkIjogICBpbnQodXNhZ2UuZ2V0KCJjYWNoZV9yZWFkIiwgMCkgb3IgMCksCiAgICAgICAgICAgICAgICAgICAgImNhY2hlX2NyZWF0ZSI6IGludCh1c2FnZS5nZXQoImNhY2hlX2NyZWF0ZSIsIDApIG9yIDApLAogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICJsYXRlbmN5X21zIjogcm91bmQobGF0ZW5jeV9tcywgMSksCiAgICAgICAgICAgICAgICAicmF3IjogeyJwcm94eSI6IFRydWUsICJiYWNrZW5kIjogc2VsZi5iYWNrZW5kLCAibW9kZWwiOiBzZWxmLm1vZGVsfSwKICAgICAgICAgICAgfQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgbGFzdF9leGMgPSBlCiAgICAgICAgICAgIGlmIGkgPCBhdHRlbXB0czoKICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAobWluKDIgKiogaSwgOCkpICAgIyAxcywgMnMsIDRzLCA4cywgOHMg4oCmCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBicmVhawogICAgcmFpc2UgU3lzdGVtRXhpdCgiTExNIHByb3h5IGZhaWxlZCBhZnRlciAlcyBhdHRlbXB0KHMpOiAlcyIKICAgICAgICAgICAgICAgICAgICAgJSAoYXR0ZW1wdHMgKyAxLCBsYXN0X2V4YykpCgpMTE1DbGllbnQuY2FsbCA9IF9wcm94aWVkX2NhbGwKCnN5cy5leGl0KGFhYi5tYWluKCkpCg=="

pathlib.Path('advisory_any_bar.py').write_bytes(base64.b64decode(SCRIPT_B64))
pathlib.Path('run_advisory.py').write_bytes(base64.b64decode(WRAPPER_B64))
skills = json.loads(base64.b64decode(SKILLS_B64).decode('utf-8'))
sd = pathlib.Path('skills'); sd.mkdir(exist_ok=True)
for name, text in skills.items():
    (sd / name).write_text(text, encoding='utf-8')
proj = json.loads(base64.b64decode(PROJECT_B64).decode('utf-8'))
for relpath, text in proj.items():
    p = pathlib.Path(relpath); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8')
print('wrote advisory_any_bar.py + run_advisory.py +',
      len(skills), 'skills +', len(proj),
      'project files (engine v5 recompute + PG-snapshot auto-detect enabled)')

## 3. Choose the bar + backend + model
`WHEN` is the bar in `DD-mon, HH:MM` form. `BACKEND` defaults to **`gemini`** — the current live-engine default (CHA-350) — and the cell above lifted your `GEMINI_API_KEY` from Colab Secrets. Flip to `nvidia`, `anthropic`, or `openrouter` if you've provisioned the corresponding key. `MODEL` defaults to **`gemini-2.5-flash`** (fast, generous free-tier); pick `gemini-2.5-pro` for higher-quality verdicts if your key has quota. `""` uses the backend's own fallback model.

In [ ]:
#@title Parameters { run: "auto" }
WHEN    = "9-Jul, 11:30"   #@param {type:"string"}
BACKEND = "gemini"         #@param ["gemini", "nvidia", "anthropic", "openrouter", "auto"]
YEAR    = 2026             #@param {type:"integer"}
# Model id. Default gemini-2.5-flash (fast, free-tier friendly). Pick
# gemini-2.5-pro for higher-quality verdicts, or "" for backend default.
# When BACKEND=nvidia, use z-ai/glm-5.2 or meta/llama-3.3-70b-instruct.
MODEL   = "gemini-2.5-flash"  #@param ["gemini-2.5-flash", "gemini-2.5-pro", "z-ai/glm-5.2", "meta/llama-3.3-70b-instruct", ""] {allow-input: true}
# Optional: paste the day sub-folder's Google Drive id to SKIP the slow
# parent-folder listing (gdown lists >600 items across ~70 day folders,
# which can OOM the Colab free tier and disconnect the runtime). The
# id is the token after 'folders/' in the day folder's Drive URL.
DAY_FOLDER_ID = ""           #@param {type:"string"}
# Legacy: route through the owner's Apps Script proxy instead of your
# own key. Leave False if you added GEMINI_API_KEY to Colab Secrets.
USE_OWNER_PROXY = False      #@param {type:"boolean"}
print('WHEN   =', WHEN)
print('BACKEND=', BACKEND, '  YEAR=', YEAR,
      '  MODEL=', MODEL or '(backend fallback)')
print('DAY_ID =', DAY_FOLDER_ID or '(auto — listing shared parent)')
print('LLM ROUTE:', 'owner Apps Script proxy' if USE_OWNER_PROXY
      else 'YOUR own key (Colab Secrets → Step 1)')

## 4. Resolve the day folder (auto, ~1 second)
Parses the bar and resolves the download folder. **New in this build:** queries Drive's lightweight `embeddedfolderview` endpoint (~1 s, ~50 KB) to get the direct list of top-level day folders under the shared parent, matches your target date by folder name, and auto-populates `DAY_FOLDER_ID`. Step 5 then downloads that folder DIRECTLY — no slow recursive listing, no OOM risk, no user interaction needed. If the auto-resolver fails (network flake / new parent id / unrecognised folder name), Step 5 falls back to the classic slow listing so nothing regresses; you can also override manually via `DAY_FOLDER_ID` in Step 3.

In [ ]:
import sys, os, argparse, re, urllib.request
sys.path.insert(0, '.')
import advisory_any_bar as aab

# Owner's Apps Script endpoint — LLM proxy + log collector.
COLLECTOR_URL = "https://script.google.com/macros/s/AKfycbx8SDCFUaAf_kxKxUOSpOnb0tsbwQ6OS-F72bjer60LQyI1-uoF7edA0XCvzCpBhtLQ/exec"

# Self-heal: rebuild WHO from the STEP-1 param even if Step 1's cell
# wasn't re-run after typing the name (a Colab `run:auto` widget update
# doesn't always fire). Falls back to '' cleanly if nothing was typed.
WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()

req = aab.parse_request(argparse.Namespace(when=WHEN, date=None,
                                           time=None, year=YEAR))
DAY_FOLDER = req.tmp_dir_name        # gdown target (e.g. 'tmp_Jun_29')


def _resolve_day_folder_id(parent_id, req_date, log_fn=print):
    """Query Drive's cheap embeddedfolderview for the DIRECT children
    of the shared parent (~1s, ~50KB), score-match every top-level folder
    name against the requested date using advisory_any_bar's own
    score_folder_name, and return the best-matching folder id. Returns
    None if the fetch fails or nothing scores > 0 — the caller then
    falls back to the slow recursive listing so nothing regresses."""
    url = f'https://drive.google.com/embeddedfolderview?id={parent_id}'
    hdrs = {'User-Agent': 'Mozilla/5.0 (compatible; trapx-notebook/1.0)'}
    try:
        rq = urllib.request.Request(url, headers=hdrs)
        with urllib.request.urlopen(rq, timeout=15) as resp:
            html = resp.read().decode('utf-8', errors='replace')
    except Exception as e:
        log_fn(f'[DRIVE-RESOLVE] embeddedfolderview fetch failed '
               f'({type(e).__name__}: {e}); Step 5 will fall back to '
               f'the slow listing.')
        return None
    # Each direct-child row looks like <a href=".../drive/folders/<ID>">
    # inside a div.flip-entry with a div.flip-entry-title carrying the
    # folder NAME. bs4 ships in Colab so parse cleanly; regex fallback
    # if bs4 ever goes missing.
    pairs = []
    try:
        import bs4
        soup = bs4.BeautifulSoup(html, 'html.parser')
        for d in soup.find_all('div', class_='flip-entry'):
            a = d.find('a', href=True)
            t = d.find('div', class_='flip-entry-title')
            if not (a and t):
                continue
            m = re.search(r'/drive/folders/([a-zA-Z0-9_-]{20,})', a['href'])
            if m:
                pairs.append((t.get_text(strip=True), m.group(1)))
    except Exception:
        pairs = []
    if not pairs:
        # Regex-only fallback: pair every folder id with the surrounding
        # anchor's title attribute or its text (name).
        pat = re.compile(
            r'<a[^>]*href="[^"]*/drive/folders/([a-zA-Z0-9_-]{20,})"[^>]*>'
            r'.*?<div class="flip-entry-title[^"]*">([^<]{1,80})</div>',
            re.DOTALL)
        pairs = [(name.strip(), fid) for (fid, name) in pat.findall(html)]
    if not pairs:
        log_fn(f'[DRIVE-RESOLVE] parsed 0 child folders from the parent '
               f'HTML ({len(html)} bytes); Step 5 will fall back.')
        return None
    # Score each folder name against the target date; keep the best.
    best_id, best_name, best_score = None, None, 0
    for name, fid in pairs:
        s = aab.score_folder_name(name, req_date)
        if s > best_score:
            best_id, best_name, best_score = fid, name, s
    if best_score <= 0:
        log_fn(f'[DRIVE-RESOLVE] no folder matched {req_date.isoformat()} '
               f'among {len(pairs)} direct children; falling back.')
        return None
    log_fn(f'[DRIVE-RESOLVE] {req_date.isoformat()} -> {best_name!r} '
           f'(id={best_id[:12]}…, score={best_score}, '
           f'{len(pairs)} folders scanned)')
    return best_id


# If the user didn't paste a DAY_FOLDER_ID in Step 3, resolve it now.
_did = str(globals().get('DAY_FOLDER_ID') or '').strip()
if not _did:
    _resolved = _resolve_day_folder_id(
        aab.DEFAULT_PARENT_FOLDER_ID, req.date)
    if _resolved:
        DAY_FOLDER_ID = _resolved   # noqa (mutated for Step 5)

print('you  :', WHO or '(enter name in STEP 1)')
print(f'date {req.iso_date}  ->  download {req.mon_dd!r}, verdict via owner LLM')
print(f'day id: '
      + (globals().get('DAY_FOLDER_ID') or '(unresolved — Step 5 will list)'))
print('log  ->  owner  external_user_logs/' + (WHO or '<you>')
      + '_<date>_<time>.log')

## 5. Run the advisory (verdict via YOUR own gemini key)
Runs the tool through the `run_advisory.py` wrapper that (a) softens `pg_connect`'s `SystemExit` so the CSV-only fallback engages when neither real PG nor a snapshot is around, and (b) delegates the LLM call to the embedded trapx `LLMClient` — which reads the API key you added to Colab Secrets in Step 1. No `--local-dir` → the script `gdown`-downloads the read-only day folder itself, INCLUDING the day's `pg_snapshot_<date>.db` if present (auto-detected by `advisory_any_bar.py`).

**Legacy owner-proxy path.** Setting `USE_OWNER_PROXY = True` below still points `TRAPX_LLM_PROXY` at the owner's Apps Script — a back-compat option for anyone still on the pre-CHA-350 owner-key flow. Default is `False` (use your own key).

**Streaming output.** Progress lines (gdown downloads, `[GATE]`, `[SKILL-COT]` drills, `[LLM] Firing…`, `[LLM] Done`) print live as they happen — the cell no longer looks frozen while advisory runs.

> **Runtime disconnected during `[DRIVE] Listing shared folder`?** Only happens if the auto-resolver in Step 4 failed AND you're on Colab free tier. Fix: open the target day folder in Drive (e.g. `Jul_09`), copy the id after `folders/` from the URL, paste it into `DAY_FOLDER_ID` in Step 3, and re-run. advisory then downloads that folder directly — no parent listing.

> Gemini `2.5-flash` normally returns in 5–15 s. `gemini-2.5-pro` can sit for 30–90 s. Either way the `[LLM] Firing…` line pausing is the LLM call, not a hang.

In [ ]:
import subprocess, sys, shlex, os

WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()
if not WHO:
    print('SKIPPED — enter your NAME or EMAIL in STEP 1 (top), then Run all again.')
else:
    cmd = [sys.executable, '-u', 'run_advisory.py', WHEN,
           '--skills-dir', 'skills', '--backend', BACKEND, '--year', str(YEAR),
           '--no-live',
           # Colab has no PG. Recent engines re-raise on pg_connect() failure
           # unless the operator opts in to CSV-only via --allow-no-db. The
           # trade-off: option-symmetry gate + run-cumulative TRAP cannot be
           # evaluated (reported in the log, not silently dropped). If the
           # day folder ships a pg_snapshot_<date>.db, advisory auto-picks it
           # up and everything runs at live-parity — this flag is inert then.
           '--allow-no-db']
    if MODEL.strip():
        cmd += ['--model', MODEL.strip()]
    # If the operator supplied a direct day-folder id, skip advisory's
    # slow parent-folder listing entirely — no gdown listing = no OOM.
    _did = str(globals().get('DAY_FOLDER_ID') or '').strip()
    if _did:
        cmd += ['--gdrive-day-id', _did]
    env = dict(os.environ)
    env['PYTHONUNBUFFERED'] = '1'   # belt+braces vs Python line buffering
    # Legacy owner-proxy path: only wire TRAPX_LLM_PROXY when the user
    # explicitly opts in via USE_OWNER_PROXY (default False). Otherwise
    # the wrapper's LLMClient.call monkey-patch stays inert and the real
    # LLMClient uses the user's own key from Step 1.
    _use_proxy = bool(globals().get('USE_OWNER_PROXY', False))
    if _use_proxy and COLLECTOR_URL.startswith('http'):
        env['TRAPX_LLM_PROXY'] = COLLECTOR_URL   # LLM -> owner's key
        print('[LLM] routed via owner Apps Script proxy (USE_OWNER_PROXY=True)')
    else:
        env.pop('TRAPX_LLM_PROXY', None)
        print('[LLM] using your own key from Colab Secrets (Step 1)')
    print('$', ' '.join(shlex.quote(c) for c in cmd), '\n', flush=True)
    # Stream stdout+stderr line-by-line so Colab shows progress live
    # (gdown downloads, [GATE], [DATA], [LLM] Firing…, [LLM] Done). Without
    # this, capture_output=True buffers everything until the subprocess
    # exits and the cell looks frozen for the full 1-5 min run.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, env=env,
                            bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    finally:
        rc = proc.wait()
    if rc != 0:
        print('\n[exit code]', rc)

## 6. The verbose log on this Colab VM
Lists the verbose log(s) written locally this session and prints the newest. Look for the `[PG-SNAPSHOT] auto-detected …` line near the top to confirm local-parity mode; if it's absent the run was CSV-only.

In [ ]:
import glob, os
WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()
if not WHO:
    print('SKIPPED — enter your NAME or EMAIL in STEP 1 (top), then Run all again.')
else:
    logs = sorted(glob.glob(f'{DAY_FOLDER}/advisory_{req.yyyymmdd}_*.log'),
                  key=os.path.getmtime)
    if logs:
        print(f'{len(logs)} verbose log(s) on this VM in {DAY_FOLDER}:')
        for p in logs:
            print('   ', os.path.basename(p), f'({os.path.getsize(p)} bytes)')
        print('\n===== newest:', os.path.basename(logs[-1]), '=====\n')
        print(open(logs[-1], encoding='utf-8').read())
    else:
        print('no verbose log in', DAY_FOLDER,
              '— did the run gate out (no pattern at that bar) or error?')

## 7. Send this run's log to the OWNER (central collection)
POSTs the newest log to the owner's collector, which saves it under `external_user_logs/<you>_<date>_<time>.log` in the **owner's** Drive. Your account never writes to Drive.

In [ ]:
import glob, os
WHO = str(globals().get('WHO') or globals().get('EXTERNAL_USER') or '').strip()
if not WHO:
    print('SKIPPED — enter your NAME or EMAIL in STEP 1 (top), then Run all again.')
else:
    src_logs = sorted(glob.glob(f'{DAY_FOLDER}/advisory_{req.yyyymmdd}_*.log'),
                      key=os.path.getmtime)
    if not src_logs:
        print('no log to send — run the previous cell first.')
    elif not COLLECTOR_URL.startswith('http'):
        print('collector URL not set — your log is on the Colab VM:')
        print('   ', os.path.abspath(src_logs[-1]))
    else:
        newest = src_logs[-1]
        body = {'user': WHO,
                'when': f'{req.yyyymmdd}_' + req.time.replace(':', ''),
                'log': open(newest, encoding='utf-8').read()}
        try:
            import requests
            resp = requests.post(COLLECTOR_URL, json=body, timeout=90).json()
            if resp.get('ok'):
                print('sent your log to the owner as:', resp.get('saved'))
            else:
                print('collector returned an error:', resp.get('error'))
        except Exception as e:
            print('could not reach collector:', type(e).__name__, e)
            print('your log is on the Colab VM:', os.path.abspath(newest))